# BEV Minimap from YOLO Tracking + Team Classification

This notebook renders selected full SoccerNet GameState validation clips as BEV/minimap videos. It uses GameState `bbox_pitch` only to calibrate homography, then projects YOLO + ByteTrack player boxes through that homography. Team labels are calibrated from the first 50 frames, then assigned by PRTReID prototype matching. Minimap rendering intentionally drops track IDs because current tracker IDs are not reliable enough for velocity/trail calculations.


In [31]:
from pathlib import Path
import copy
import csv
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from types import SimpleNamespace

import cv2
import numpy as np
import torch
from tqdm.auto import tqdm


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    marker_sets = [
        ("data", "notebooks", "outputs"),
        ("data", "outputs"),
        ("README.md", "notebooks"),
    ]
    for candidate in [current, *current.parents]:
        for markers in marker_sets:
            if all((candidate / marker).exists() for marker in markers):
                return candidate
    return current


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

GAMESTATE_VALID_DIR = PROJECT_ROOT / "data" / "SoccerNetGS" / "valid"
YOLO_WEIGHTS = PROJECT_ROOT / "outputs" / "detect_player" / "runs" / "E1_yolo_fullframe_img960" / "weights" / "best.pt"
BEV_MINIMAP_PATH = PROJECT_ROOT / "bev" / "bev_minimap.py"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "bev" / "yolo_tracking_team"

N_SCENES = 2
MAX_FRAMES = None  # None = render full clip; set 10/150 for quick tests.
IMG_SIZE = 960
TRACK_CONF = 0.05
TRACK_IOU = 0.60
REID_BATCH_SIZE = 32
SHOW_TRACK_IDS = False
TRAIL_FRAMES = 0
INCLUDE_TRACK_IDS_IN_MINIMAP_LABELS = False
POSITION_SMOOTHING = 1.0
MAX_POSITION_STEP_M = 0.0
TEAM_CALIBRATION_FRAMES = 50
TRANSCODE_H264 = True

DEVICE = "cuda:0" if torch.cuda.is_available() and torch.cuda.device_count() > 0 else "cpu"
print("Project root:", PROJECT_ROOT)
print("GameState valid:", GAMESTATE_VALID_DIR, "exists=", GAMESTATE_VALID_DIR.exists())
print("YOLO weights:", YOLO_WEIGHTS, "exists=", YOLO_WEIGHTS.exists())
print("BEV minimap script:", BEV_MINIMAP_PATH, "exists=", BEV_MINIMAP_PATH.exists())
print("Device:", DEVICE)

if DEVICE.startswith("cuda"):
    print("CUDA GPU:", torch.cuda.get_device_name(0))


Project root: C:\Users\CPU13374\Downloads\SoccerNet
GameState valid: C:\Users\CPU13374\Downloads\SoccerNet\data\SoccerNetGS\valid exists= True
YOLO weights: C:\Users\CPU13374\Downloads\SoccerNet\outputs\detect_player\runs\E1_yolo_fullframe_img960\weights\best.pt exists= True
BEV minimap script: C:\Users\CPU13374\Downloads\SoccerNet\bev\bev_minimap.py exists= True
Device: cuda:0
CUDA GPU: NVIDIA GeForce RTX 5060 Ti


In [32]:
# Imports that depend on the project paths above.
from ultralytics import YOLO

from detect_player.config import PlayerTeamClassifierConfig
from detect_player.reid_backend import PRTReIDBackend
from detect_player.team_assignment import assign_roles, assign_teams_temporal_prototypes, resolve_yolo_classes, yolo_model_has_roles

spec = importlib.util.spec_from_file_location("bev", BEV_MINIMAP_PATH)
bev_minimap = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = bev_minimap
spec.loader.exec_module(bev_minimap)


In [33]:
def find_gamestate_scenes(valid_dir: Path):
    return sorted(path.parent for path in valid_dir.rglob("Labels-GameState.json"))


scene_dirs = find_gamestate_scenes(GAMESTATE_VALID_DIR)
if not scene_dirs:
    raise FileNotFoundError(
        "No GameState valid scenes found. Run notebooks/00_download_soccernet_tracking.ipynb "
        "GameState valid cells first. Expected data/SoccerNetGS/valid/<scene>/Labels-GameState.json."
    )
if not YOLO_WEIGHTS.exists():
    raise FileNotFoundError(f"Missing YOLO checkpoint: {YOLO_WEIGHTS}")
if not BEV_MINIMAP_PATH.exists():
    raise FileNotFoundError(f"Missing BEV minimap script: {BEV_MINIMAP_PATH}")

selected_scenes = scene_dirs[-N_SCENES:]
print(f"Found {len(scene_dirs)} valid scenes")
print("Selected scenes:")
for scene in selected_scenes:
    print("-", scene.name)


Found 58 valid scenes
Selected scenes:
- SNGS-095
- SNGS-096


## Helpers

The model labels intentionally do not store `bbox_pitch`. Homography is saved separately from the original GameState labels, then the generated model bboxes are projected by `bev_minimap.py` via `--homography-csv`.


In [34]:
def scene_labels_path(scene_dir: Path) -> Path:
    labels_path = scene_dir / "Labels-GameState.json"
    if not labels_path.exists():
        raise FileNotFoundError(labels_path)
    return labels_path


def load_labels(labels_path: Path) -> dict:
    return json.loads(labels_path.read_text(encoding="utf-8"))


def select_scene_frames(labels_path: Path, max_frames=None):
    frames, annotations_by_image, info = bev_minimap.load_annotations(labels_path)
    selected_frames = bev_minimap.filter_frames(frames, start_frame=None, end_frame=None, max_frames=max_frames)
    if not selected_frames:
        raise RuntimeError(f"No frames selected from {labels_path}")
    return selected_frames, annotations_by_image, info


def source_image_path(labels_path: Path, info: dict, frame) -> Path:
    return labels_path.parent / str(info.get("im_dir") or "img1") / frame.file_name


def xyxy_to_bbox_image(x1, y1, x2, y2):
    return {"x": float(x1), "y": float(y1), "w": float(x2 - x1), "h": float(y2 - y1)}


def role_to_category_id(role: str, yolo_class_name: str):
    role = (role or "").lower()
    yolo_class_name = (yolo_class_name or "").lower()
    if role == "goalkeeper" or yolo_class_name == "goalkeeper":
        return 2
    if role == "referee" or yolo_class_name == "referee":
        return 3
    if role == "player" or yolo_class_name == "player":
        return 1
    return None


def category_role(category_id: int) -> str:
    return {1: "player", 2: "goalkeeper", 3: "referee"}.get(int(category_id), "unknown")


In [35]:
def save_homography_csv(labels_path: Path, selected_frames, annotations_by_image, output_csv: Path):
    args = SimpleNamespace(
        homography_csv=None,
        min_homography_points=4,
        ransac_threshold=3.0,
        include_referees_in_homography=False,
        homography_footpoints="bottom-line",
        no_reuse_last_homography=False,
        homography_smoothing=0.18,
        max_homography_jump_m=10.0,
    )
    _, homography_rows = bev_minimap.build_homographies(selected_frames, annotations_by_image, args)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    bev_minimap.write_homographies(output_csv, homography_rows)
    valid_h = sum(1 for _, h in homography_rows if h.matrix is not None)
    print(f"Homography frames: {valid_h}/{len(homography_rows)} -> {output_csv}")
    return valid_h


In [36]:
def track_scene_with_yolo(labels_path: Path, selected_frames, info: dict):
    model = YOLO(str(YOLO_WEIGHTS))
    yolo_class_names = {int(key): str(value) for key, value in model.names.items()}
    yolo_classes = resolve_yolo_classes(yolo_class_names, requested=None)
    yolo_has_roles = yolo_model_has_roles(yolo_class_names)

    metadata = []
    crops = []

    for image_idx, frame in enumerate(tqdm(selected_frames, desc=f"Tracking {labels_path.parent.name}")):
        frame_path = source_image_path(labels_path, info, frame)
        image = cv2.imread(str(frame_path))
        if image is None:
            print(f"Skipping unreadable frame: {frame_path}")
            continue

        result = model.track(
            image,
            tracker="bytetrack.yaml",
            persist=True,
            imgsz=IMG_SIZE,
            conf=TRACK_CONF,
            iou=TRACK_IOU,
            device=DEVICE,
            verbose=False,
        )[0]
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue

        xyxy = boxes.xyxy.cpu().numpy()
        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()
        track_ids = boxes.id.cpu().numpy().astype(int) if boxes.id is not None else np.full(len(boxes), -1, dtype=int)
        height, width = image.shape[:2]

        for bbox, class_id, conf, track_id in zip(xyxy, cls_ids, confs, track_ids):
            class_id = int(class_id)
            if class_id not in yolo_classes:
                continue
            x1, y1, x2, y2 = bbox.tolist()
            x1 = max(0, min(width - 1, int(round(x1))))
            y1 = max(0, min(height - 1, int(round(y1))))
            x2 = max(0, min(width, int(round(x2))))
            y2 = max(0, min(height, int(round(y2))))
            if x2 <= x1 or y2 <= y1:
                continue

            crop = image[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            crops.append(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            metadata.append(
                {
                    "image_idx": image_idx,
                    "image_id": frame.image_id,
                    "frame_number": frame.frame_number,
                    "file_name": frame.file_name,
                    "bbox_xyxy": [x1, y1, x2, y2],
                    "bbox_image": xyxy_to_bbox_image(x1, y1, x2, y2),
                    "detection_confidence": float(conf),
                    "track_id": int(track_id) if int(track_id) >= 0 else None,
                    "yolo_class_id": class_id,
                    "yolo_class_name": yolo_class_names.get(class_id, "unknown").lower(),
                }
            )

    if not crops:
        return [], metadata

    config = PlayerTeamClassifierConfig.from_project_defaults(
        project_root=PROJECT_ROOT,
        device=DEVICE,
        yolo_weights=YOLO_WEIGHTS,
        reid_batch_size=REID_BATCH_SIZE,
    )
    reid_backend = PRTReIDBackend(config)
    embeddings, role_scores = reid_backend.extract_features(crops)
    results = assign_roles(
        metadata=metadata,
        embeddings=embeddings,
        role_scores=role_scores,
        yolo_has_roles=yolo_has_roles,
    )
    assign_teams_temporal_prototypes(
        results,
        metadata,
        calibration_frames=TEAM_CALIBRATION_FRAMES,
        use_track_pooling=True,
    )
    return results, metadata


In [37]:
def build_model_labels(original_labels: dict, labels_path: Path, selected_frames, results, metadata, output_dir: Path):
    labels_out = copy.deepcopy(original_labels)
    image_by_id = {
        str(image.get("image_id", index)): image
        for index, image in enumerate(original_labels.get("images", []), start=1)
    }
    selected_images = []
    for frame in selected_frames:
        image_info = copy.deepcopy(image_by_id.get(str(frame.image_id), {}))
        if not image_info:
            image_info = {"image_id": frame.image_id, "file_name": frame.file_name, "width": frame.width, "height": frame.height}
        selected_images.append(image_info)

    info = copy.deepcopy(original_labels.get("info", {}))
    original_im_dir = (labels_path.parent / str(info.get("im_dir") or "img1")).resolve()
    try:
        info["im_dir"] = os.path.relpath(original_im_dir, start=output_dir.resolve()).replace("\\", "/")
    except ValueError:
        info["im_dir"] = str(original_im_dir)
    info["name"] = f"{info.get('name') or labels_path.parent.name}_yolo_tracking_team"

    annotations = []
    ann_id = 1
    for result, meta in zip(results, metadata):
        category_id = role_to_category_id(result.role, meta.get("yolo_class_name", ""))
        if category_id is None:
            continue
        team = result.side_label if category_id in (1, 2) else None
        attributes = {"role": category_role(category_id)}
        if team is not None:
            attributes["team"] = team
        annotations.append(
            {
                "id": ann_id,
                "image_id": meta["image_id"],
                "category_id": category_id,
                "track_id": meta.get("track_id") if INCLUDE_TRACK_IDS_IN_MINIMAP_LABELS else None,
                "bbox_image": meta["bbox_image"],
                "score": float(result.detection_confidence),
                "attributes": attributes,
            }
        )
        ann_id += 1

    labels_out["info"] = info
    labels_out["images"] = selected_images
    labels_out["annotations"] = annotations
    return labels_out


In [38]:
def write_team_assignment_debug(path: Path, results, metadata):
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "frame_number",
        "image_id",
        "track_id",
        "role",
        "team_id",
        "side_label",
        "assignment_source",
        "distance_to_left",
        "distance_to_right",
    ]
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for result, meta in zip(results, metadata):
            writer.writerow(
                {
                    "frame_number": meta.get("frame_number"),
                    "image_id": meta.get("image_id"),
                    "track_id": meta.get("track_id") if INCLUDE_TRACK_IDS_IN_MINIMAP_LABELS else None,
                    "role": result.role,
                    "team_id": result.team_id,
                    "side_label": result.side_label,
                    "assignment_source": getattr(result, "team_assignment_source", None),
                    "distance_to_left": getattr(result, "distance_to_left", None),
                    "distance_to_right": getattr(result, "distance_to_right", None),
                }
            )
    print("Team assignment debug:", path)


In [39]:
def derived_output_path(output_path: Path, suffix: str) -> Path:
    return output_path.with_name(f"{output_path.stem}_{suffix}{output_path.suffix}")


def find_ffmpeg_executable():
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg:
        return ffmpeg
    try:
        import imageio_ffmpeg

        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None


def transcode_to_h264(video_path: Path):
    video_path = Path(video_path)
    if not video_path.exists():
        print(f"Skip H.264 transcode, missing video: {video_path}")
        return video_path

    ffmpeg = find_ffmpeg_executable()
    if ffmpeg is None:
        raise RuntimeError(
            "H.264 transcode needs ffmpeg. Install one of these, then rerun the render cell:\n"
            "  1) Windows: winget install Gyan.FFmpeg\n"
            "  2) Python: %pip install imageio-ffmpeg"
        )

    tmp_path = video_path.with_name(f"{video_path.stem}_h264_tmp{video_path.suffix}")
    cmd = [
        ffmpeg,
        "-y",
        "-i",
        str(video_path),
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-movflags",
        "+faststart",
        "-preset",
        "veryfast",
        "-crf",
        "23",
        "-an",
        str(tmp_path),
    ]
    completed = subprocess.run(cmd, text=True, capture_output=True)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"ffmpeg H.264 transcode failed for {video_path}")
    tmp_path.replace(video_path)
    print("H.264 video:", video_path)
    return video_path


def transcode_minimap_outputs_to_h264(output_video: Path):
    if not TRANSCODE_H264:
        print("TRANSCODE_H264=False; keeping original OpenCV mp4v videos.")
        return
    videos = [
        Path(output_video),
        derived_output_path(Path(output_video), "original"),
        derived_output_path(Path(output_video), "side_by_side"),
    ]
    for video in videos:
        transcode_to_h264(video)


def render_minimap(model_labels_path: Path, homography_csv: Path, output_video: Path, max_frames: int):
    cmd = [
        sys.executable,
        str(BEV_MINIMAP_PATH),
        "--labels",
        str(model_labels_path),
        "--homography-csv",
        str(homography_csv),
        "--output",
        str(output_video),
        "--max-frames",
        str(max_frames),
        "--debug",
        "--trail-frames",
        str(TRAIL_FRAMES),
        "--position-smoothing",
        str(POSITION_SMOOTHING),
        "--max-position-step-m",
        str(MAX_POSITION_STEP_M),
    ]
    if SHOW_TRACK_IDS:
        cmd.append("--show-track-ids")
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"bev_minimap.py failed with exit code {completed.returncode}")
    transcode_minimap_outputs_to_h264(output_video)
    return output_video


## Run Sample Scenes

Default settings render the full selected clips. For a very fast smoke test, set `N_SCENES = 1` and `MAX_FRAMES = 10` in the config cell above, then run from the top.


In [40]:
summaries = []
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for scene_dir in selected_scenes:
    scene_name = scene_dir.name
    scene_output = OUTPUT_ROOT / scene_name
    scene_output.mkdir(parents=True, exist_ok=True)

    labels_path = scene_labels_path(scene_dir)
    original_labels = load_labels(labels_path)
    selected_frames, annotations_by_image, info = select_scene_frames(labels_path, MAX_FRAMES)

    print("\n" + "=" * 80)
    print("Scene:", scene_name)
    print("Frames:", len(selected_frames))

    homography_csv = scene_output / "homographies.csv"
    valid_h = save_homography_csv(labels_path, selected_frames, annotations_by_image, homography_csv)
    if valid_h == 0:
        raise RuntimeError(f"No valid homographies for {scene_name}; cannot render model bbox minimap.")

    results, metadata = track_scene_with_yolo(labels_path, selected_frames, info)
    # print(f"Model detections with crops: {len(results)}")
    if not results:
        raise RuntimeError(f"YOLO/ByteTrack produced no detections for {scene_name}")

    model_labels = build_model_labels(original_labels, labels_path, selected_frames, results, metadata, scene_output)
    model_labels_path = scene_output / "model_labels.json"
    model_labels_path.write_text(json.dumps(model_labels, indent=2), encoding="utf-8")
    # print("Model labels:", model_labels_path)

    debug_csv_path = scene_output / "team_assignment_debug.csv"
    write_team_assignment_debug(debug_csv_path, results, metadata)

    minimap_path = scene_output / "minimap.mp4"
    render_minimap(model_labels_path, homography_csv, minimap_path, len(selected_frames))

    summary = {
        "scene": scene_name,
        "frames": len(selected_frames),
        "homography_frames": valid_h,
        "detections": len(model_labels.get("annotations", [])),
        "model_labels": str(model_labels_path),
        "homographies": str(homography_csv),
        "team_debug": str(debug_csv_path),
        "minimap": str(minimap_path),
        "original": str(scene_output / "minimap_original.mp4"),
        "side_by_side": str(scene_output / "minimap_side_by_side.mp4"),
    }
    summaries.append(summary)

# summaries



Scene: SNGS-095
Frames: 750
Homography frames: 750/750 -> C:\Users\CPU13374\Downloads\SoccerNet\outputs\bev\yolo_tracking_team\SNGS-095\homographies.csv


Tracking SNGS-095: 100%|██████████| 750/750 [00:13<00:00, 56.75it/s]


Overwriting current config with config loaded from C:\Users\CPU13374\Downloads\SoccerNet\models\reid\prtreid-soccernet-baseline.pth.tar
Diff from default config :
{'dim_reduce_output': 256,
 'hrnet_pretrained_path': 'C:\\Users\\CPU13374\\Downloads\\SoccerNet\\models\\reid',
 'load_config': True,
 'mask_filtering_testing': False,
 'preprocess': 'id',
 'test_embeddings': "['globl']"}
building model on device cuda:0
=> init weights from normal distribution
Loading pretrained ImageNet HRNet32 model at C:\Users\CPU13374\Downloads\SoccerNet\models\reid\hrnetv2_w32_imagenet_pretrained.pth
=> loading pretrained model C:\Users\CPU13374\Downloads\SoccerNet\models\reid\hrnetv2_w32_imagenet_pretrained.pth
Successfully loaded pretrained weights from "C:\Users\CPU13374\Downloads\SoccerNet\models\reid\prtreid-soccernet-baseline.pth.tar"
** The following layers are discarded due to unmatched keys or layer size: ['global_identity_classifier.classifier.weight', 'background_identity_classifier.classifier

Tracking SNGS-096: 100%|██████████| 750/750 [00:14<00:00, 53.55it/s]


Overwriting current config with config loaded from C:\Users\CPU13374\Downloads\SoccerNet\models\reid\prtreid-soccernet-baseline.pth.tar
Diff from default config :
{'dim_reduce_output': 256,
 'hrnet_pretrained_path': 'C:\\Users\\CPU13374\\Downloads\\SoccerNet\\models\\reid',
 'load_config': True,
 'mask_filtering_testing': False,
 'preprocess': 'id',
 'test_embeddings': "['globl']"}
building model on device cuda:0
=> init weights from normal distribution
Loading pretrained ImageNet HRNet32 model at C:\Users\CPU13374\Downloads\SoccerNet\models\reid\hrnetv2_w32_imagenet_pretrained.pth
=> loading pretrained model C:\Users\CPU13374\Downloads\SoccerNet\models\reid\hrnetv2_w32_imagenet_pretrained.pth
Successfully loaded pretrained weights from "C:\Users\CPU13374\Downloads\SoccerNet\models\reid\prtreid-soccernet-baseline.pth.tar"
** The following layers are discarded due to unmatched keys or layer size: ['global_identity_classifier.classifier.weight', 'background_identity_classifier.classifier